In [1]:
import torch
import os
from src.cnn.SleepNet.model import SleepNet
from src.cnn.SleepNet.predictor import SleepNetPredictor
from src.cnn.utils.data_loader import make_loaders
from src.cnn.utils.trainer import Trainer
from src.utils.config_readers.ReaderConfig import ReaderConfig
from src.utils.config_readers.ViBEConfig import ViBEConfig
from src.video_processing.input_reader import reader
import src.utils.basic_functions.BasicFunctions as BasicFunctions
import numpy as np

from src.video_processing.vibe_extractor.vibe_extractor import pybind_process_three_channels

In [2]:
ROOT_DIR = os.path.split(os.environ['VIRTUAL_ENV'])[0]
path_to_weights = os.path.join(ROOT_DIR, 'data', 'models','SleepNet', 'SleepNet_v5.pth')
path_to_video = os.path.join(ROOT_DIR, 'inputs', 'kesha', 'video.mp4')
path_to_config = os.path.join(ROOT_DIR, 'config', 'config.yml')

In [3]:
predictor = SleepNetPredictor(path_to_weights)

In [4]:
reader_config = ReaderConfig.from_yaml(path_to_config)
frames = reader.rsv_read_all(path_to_video, reader_config)

print(frames.shape)

(232, 180, 240, 3)


In [5]:
vibe_config = ViBEConfig.from_yaml(path_to_config)
masks = pybind_process_three_channels(frames, vibe_config)

print(masks.shape)

(232, 180, 240)


In [6]:
masks_64 = np.empty(shape=(masks.shape[0],64,64),dtype=np.uint8)

for i in range(frames.shape[0]):
    masks_64[i] = BasicFunctions.get_64pix_mask(masks[i])
    
print(masks_64.shape)

(232, 64, 64)


In [7]:
input_tensor = torch.from_numpy(masks_64).float()

print(input_tensor.shape)

torch.Size([232, 64, 64])


In [8]:
results = predictor.predict_single(masks_64[0]) 
print(results)

2


In [9]:
results = predictor.predict_batch(masks_64) 
print(results)

[2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2]
